# Computer Use con Claude: el bucle de control

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/computer-use-operator/01-computer-use-intro.ipynb)

> ⚠️ **Nota:** Este notebook requiere un entorno con display virtual (Docker + xvfb). Ver el README del bloque para configuración. Las celdas de código muestran la arquitectura y pueden ejecutarse parcialmente en Colab.

In [ ]:
!pip install anthropic pillow -q

## 1. Herramientas de Computer Use

Claude dispone de 3 herramientas en modo Computer Use:

In [ ]:
# Definición de herramientas para Computer Use
TOOLS_COMPUTER_USE = [
    {
        'type': 'computer_20250124',
        'name': 'computer',
        'display_width_px': 1280,
        'display_height_px': 800,
        'display_number': 1,
    },
    {
        'type': 'bash_20250124',
        'name': 'bash',
    },
    {
        'type': 'text_editor_20250429',
        'name': 'str_replace_based_edit_tool',
    },
]

acciones_disponibles = {
    'computer': ['screenshot', 'left_click', 'right_click', 'double_click', 'type', 'key', 'mouse_move', 'scroll', 'drag'],
    'bash': ['ejecutar comandos de shell'],
    'text_editor': ['view', 'create', 'str_replace', 'insert'],
}

for herramienta, acciones in acciones_disponibles.items():
    print(f"🔧 {herramienta}: {', '.join(acciones)}")

## 2. El bucle de control

In [ ]:
import anthropic

client = anthropic.Anthropic()

def agente_computer_use_simulado(instruccion: str, max_iter: int = 5):
    """
    Versión simplificada del bucle de control.
    En producción, las acciones reales se ejecutan sobre el display.
    """
    mensajes = [{'role': 'user', 'content': instruccion}]

    print(f'🎯 Instrucción: {instruccion}')
    print('=' * 60)

    for i in range(max_iter):
        response = client.beta.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=1024,
            tools=TOOLS_COMPUTER_USE,
            messages=mensajes,
            betas=['computer-use-2025-01-24'],
        )

        mensajes.append({'role': 'assistant', 'content': response.content})
        tool_uses = [b for b in response.content if b.type == 'tool_use']

        if not tool_uses:
            texto = next((b.text for b in response.content if b.type == 'text'), '')
            print(f'✅ Completado en {i+1} iter: {texto[:200]}')
            return texto

        for tu in tool_uses:
            accion = tu.input.get('action', str(tu.input)[:50])
            print(f'[Iter {i+1}] {tu.name} → {accion}')

        # Simular resultados de herramientas (en producción: ejecutar realmente)
        resultados = [{
            'type': 'tool_result',
            'tool_use_id': tu.id,
            'content': 'Acción ejecutada (simulado)',
        } for tu in tool_uses]
        mensajes.append({'role': 'user', 'content': resultados})

# Demo: Claude planifica las acciones para completar la tarea
agente_computer_use_simulado(
    'Abre el navegador, ve a google.com y busca "inteligencia artificial"',
    max_iter=3,
)

## 3. Análisis visual de screenshots

Claude puede analizar screenshots incluso sin ejecutar Computer Use.

In [ ]:
import base64
from pathlib import Path

def analizar_screenshot(imagen_path: str, pregunta: str) -> str:
    """Analiza un screenshot con Claude Vision."""
    imagen_b64 = base64.standard_b64encode(Path(imagen_path).read_bytes()).decode()
    
    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=512,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': imagen_b64}},
                {'type': 'text', 'text': pregunta},
            ],
        }],
    )
    return response.content[0].text

# Ejemplo: analizar un screenshot guardado
# resultado = analizar_screenshot('/tmp/screenshot.png', '¿Qué aplicaciones están abiertas?')
print('Función lista. Usa analizar_screenshot(ruta, pregunta) con un screenshot real.')

## Resumen

- El bucle: **screenshot → análisis → acción → screenshot**
- Modelo recomendado: **claude-sonnet-4-6**
- Requiere entorno Docker con xvfb para producción
- Beta header: `computer-use-2025-01-24`